# Explore constant phase offset

In [ ]:
from n23_fluctuations_ragland import tools
import nescafe23 as n23
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
dss = n23.open_receptions()

In [ ]:
wavefront_values = tools.get_wavefront_values(dss, remove_array_tilt=True)

In [ ]:
wf_vals_arr = []

ray_id_omit = ['-15b', '-14b', '-6b', '-13b', '-11b', '+11b', '-10b']
for ray_id in wavefront_values.keys():

    if ray_id in ray_id_omit:
        continue
    wf_vals_arr.append(wavefront_values[ray_id])

wf_vals_arr = np.array(wf_vals_arr)

In [ ]:
phase =np.unwrap(np.angle(wf_vals_arr), axis=2)

In [ ]:
for k in range(len(phase[:,0,0])):
    for i in range(9):
        plt.subplot(3,3,i+1)
        plt.plot(phase[k, i, :] - np.mean(phase[k, i, :],), label=list(wavefront_values.keys())[k])

#plt.legend()


In [ ]:
wavefront_values.keys().values

In [ ]:
# Auto-covariance of phase vs. depth lag, estimated per ray by averaging over all
# hydrophone pairs at each lag (assumes statistical homogeneity in depth) and over transmissions.
dz = 9.0  # lower-array hydrophone spacing [m]
remove_constant_offset = False  # also remove each transmission's depth-mean phase

ray_ids = [ray_id for ray_id in ['-7', '+7', '-6', '+6']]

# phase fluctuation: remove the transmission-mean phase at each depth
dphi = phase - np.nanmean(phase, axis=1, keepdims=True)
if remove_constant_offset:
    dphi = dphi - np.nanmean(dphi, axis=2, keepdims=True)

n_z = dphi.shape[2]
lags = np.arange(n_z)
acov = np.full((dphi.shape[0], n_z), np.nan)
for lag in lags:
    acov[:, lag] = np.nanmean(dphi[:, :, : n_z - lag] * dphi[:, :, lag:], axis=(1, 2))

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for k, ray_id in enumerate(ray_ids):
    axs[0].plot(lags * dz, acov[k], label=ray_id)
    axs[1].plot(lags * dz, acov[k] / acov[k, 0], label=ray_id)
#axs[1].plot(lags * dz, np.mean(acov / acov[:, :1], axis=0), 'k', lw=3, label='ray mean')

axs[0].set_ylabel(r'auto-covariance of phase [rad$^2$]')
axs[1].set_ylabel('normalized auto-covariance of phase')
for ax in axs:
    ax.set_xlabel(r'depth lag $\Delta z$ [m]')
    ax.axhline(0, color='gray', lw=0.5)
    ax.grid(alpha=0.3)
axs[1].legend(ncol=2, fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()